> deepseek r1 vs. qwen3
- DeepSeek-R1-Distill-Qwen: 1.5/7/14/32, llama3: 8/70
    - “基于数据的 SFT” (SFT-only)： 采用经典的**有监督微调（SFT）**模式。即利用大模型生成高质量数据（800k samples），直接对小模型进行微调，不包含强化学习（RL）或 Logits 对齐 。
    - DeepSeek 实验发现，直接蒸馏大模型的输出，比让小模型自己从头进行大规模强化学习（RL）效果更好。例如，蒸馏后的 DeepSeek-R1-Distill-Qwen-32B 性能显著优于自己跑 RL 的 DeepSeek-R1-Zero-Qwen-32B 。
- qwen3
    - Qwen3 的蒸馏是“基于策略的 Logits 对齐”： 采用Strong-to-Weak 两阶段蒸馏。不仅包含离策略（Off-policy）的数据学习，还引入了在线策略（On-policy）蒸馏，通过最小化 KL 散度（KL Divergence）让学生模型在生成过程中实时对齐教师模型的概率分布 。
    - 教师模型 (Teacher): Qwen3-32B 或 Qwen3-235B-A22B 。学生模型 (Students): Qwen3 系列轻量级模型 (0.6B - 14B, 30B-A3B) 。

### qwen3 的两阶段

- 第一阶段：离策略蒸馏 (Off-policy Distillation)
    - 数据准备：直接使用教师模型（Teacher Models）生成的输出作为训练数据 。
    - 混合模式：数据中同时包含了 /think（思考模式） 和 /no_think（非思考模式） 的响应样本 。
    - 训练目标：对学生模型（Student Model）进行监督微调（SFT）。
        - 更重要的是，让小模型学会识别和响应模式切换指令（即根据 prompt 中的 /think 或 /no_think 标签来决定是否进行长思维链推理）。
$$
D_{KL}(P_{\text{Teacher}} || P_{\text{Student}})
$$
- 第二阶段：在线策略蒸馏 (On-policy Distillation)
    - 学生生成 (Student Generation)：采样一批 Prompts，让经过第一阶段训练的学生模型在线（On-policy） 生成响应序列 。这意味着数据不是预先固定的，而是由学生模型当前的策略实时产生的。
    - 模式覆盖：学生模型会在 /think 或 /no_think 两种模式下生成响应 。
    - Logits 对齐 (Logits Alignment)：将学生模型生成的序列输入给强大的教师模型（Qwen3-32B 或 Qwen3-235B-A22B），获取教师模型对这些 token 的概率分布（Logits）。
    - 优化目标：通过最小化 KL 散度 (KL Divergence)，迫使学生模型的输出概率分布尽可能接近教师模型 。